In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    # !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install transformers==4.57.1
!pip install --no-deps trl==0.22.2

In [1]:
import os
from PIL import Image

In [2]:
from unsloth import FastVisionModel # FastLanguageModel for LLMs
import torch

model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/Qwen3-VL-8B-Instruct-unsloth-bnb-4bit",
    load_in_4bit = True, # Use 4bit to reduce memory use. False for 16bit LoRA.
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for long context
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/home/ubuntu/emonarrify/18789_venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.6: Fast Qwen3_Vl patching. Transformers: 4.57.1.
   \\   /|    NVIDIA L40S. Num GPUs = 1. Max memory: 44.392 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards: 100%|██████████| 2/2 [00:57<00:00, 28.87s/it]


In [3]:
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = True, # False if not finetuning vision layers
    finetune_language_layers   = True, # False if not finetuning language layers
    finetune_attention_modules = True, # False if not finetuning attention layers
    finetune_mlp_modules       = True, # False if not finetuning MLP layers

    r = 16,           # The larger, the higher the accuracy, but might overfit
    lora_alpha = 16,  # Recommended alpha == r at least
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
    # target_modules = "all-linear", # Optional now! Can specify a list if needed
)

In [6]:
JSONL_PATH = "../../data/phase1_vist_train_new.jsonl"

In [7]:
from datasets import load_dataset
dataset = load_dataset("json", data_files=JSONL_PATH)["train"]
dataset = dataset.filter(lambda x: os.path.exists(os.path.expanduser(x["image_path"])))

In [8]:
dataset

Dataset({
    features: ['image_path', 'narrative_text', 'emotion_label'],
    num_rows: 2765
})

In [9]:
dataset[2]["image_path"]

'/home/ubuntu/emonarrify/images/val/2636669693.jpg'

In [10]:
image = Image.open(dataset[2]["image_path"])

In [11]:
dataset[2]["narrative_text"]

'people were dressed up in their favorite red , white , and blue .'

In [12]:
dataset[2]["emotion_label"]

'neutral'

In [13]:
# instruction = "Write the LaTeX representation for this image."
instruction = (
    "You must use the image to answer.\n"
    # "If the answer cannot be seen in the image, do not say it.\n\n"
    "First describe the image briefly.\n"
    # "Then predict emotion.\n\n"
    "Then choose one emotion from: neutral, happy, angry, sad, surprise.\n\n"
)

def convert_to_conversation(sample):
    conversation = [
        { "role": "user",
          "content" : [
            {"type" : "text",  "text"  : instruction},
            {"type" : "image", "image" : Image.open(sample["image_path"])} ]
        },
        { "role" : "assistant",
          "content" : [
                {"type": "text", "text": (
                    f"Description: {sample['narrative_text']}\n"
                    f"Emotion: {sample['emotion_label']}"
                )}]
        },
    ]
    return { "messages" : conversation }
pass

In [14]:
converted_dataset = [convert_to_conversation(sample) for sample in dataset]

In [15]:
converted_dataset[0]

{'messages': [{'role': 'user',
   'content': [{'type': 'text',
     'text': 'You must use the image to answer.\nFirst describe the image briefly.\nThen choose one emotion from: neutral, happy, angry, sad, surprise.\n\n'},
    {'type': 'image',
     'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=2608x1952>}]},
  {'role': 'assistant',
   'content': [{'type': 'text',
     'text': 'Description: yum , some even better beats and onions are right here .\nEmotion: neutral'}]}]}

In [16]:
FastVisionModel.for_inference(model) # Enable for inference!

image = Image.open(dataset[2]["image_path"])
# instruction = "Write the LaTeX representation for this image."

messages = [
    {"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": instruction}
    ]}
]

input_text = tokenizer.apply_chat_template(messages, add_generation_prompt = True)
inputs = tokenizer(
    image,
    input_text,
    add_special_tokens = False,
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 128,
                   use_cache = True, temperature = 1.5, min_p = 0.1)

A woman at a patriotic event, wearing a red, white, and blue star-patterned outfit, a fringed patriotic hat, and holding small American flags. She's smiling and adorned with beads and a USA pin.

happy<|im_end|>


In [17]:
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

FastVisionModel.for_training(model) # Enable for training!

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    data_collator = UnslothVisionDataCollator(model, tokenizer), # Must use!
    train_dataset = converted_dataset,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 30,
        # num_train_epochs = 1, # Set this instead of max_steps for full training runs
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",     # For Weights and Biases

        # You MUST put the below items for vision finetuning:
        remove_unused_columns = False,
        dataset_text_field = "",
        dataset_kwargs = {"skip_prepare_dataset": True},
        max_length = 2048,
    ),
)

Unsloth: Model does not have a default image size - using 512


In [18]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,765 | Num Epochs = 1 | Total steps = 30
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 51,346,944 of 8,818,470,640 (0.58% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,5.082800
2,5.104200
3,4.829900
4,4.472200
5,3.774700
6,2.983700
7,2.439100
8,2.083700
9,1.875100
10,1.650400


In [23]:
JSONL_PATH = "../../data/phase1_vist_test_new.jsonl"

dataset = load_dataset("json", data_files=JSONL_PATH)["train"]
dataset = dataset.filter(lambda x: os.path.exists(os.path.expanduser(x["image_path"])))

In [24]:
dataset

Dataset({
    features: ['image_path', 'narrative_text', 'emotion_label'],
    num_rows: 56
})

In [27]:
import re

def parse_output(text):
    # 移掉特殊 token
    text = text.replace("<|im_end|>", "").strip()

    desc_match = re.search(r"Description:\s*(.*)", text)
    emo_match  = re.search(r"Emotion:\s*(.*)", text)

    description = desc_match.group(1).strip() if desc_match else ""
    emotion = emo_match.group(1).strip() if emo_match else ""

    return description, emotion

In [36]:
import json


correct = 0
total = 0

def run_inference(sample):
    global correct, total
    image = Image.open(sample["image_path"])

    messages = [
        {"role": "user", "content": [
            {"type": "image"},
            {"type": "text", "text": instruction}
        ]}
    ]

    input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
    inputs = tokenizer(
        image,
        input_text,
        add_special_tokens = False,
        return_tensors = "pt",
    ).to("cuda")

    from transformers import TextStreamer
    text_streamer = TextStreamer(tokenizer, skip_prompt = True)
    output_ids = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 128,
                   use_cache = True, temperature = 1.5, min_p = 0.1)

    output_text = tokenizer.decode(output_ids[0], skip_special_tokens=False)

    description, emotion = parse_output(output_text)

    if emotion == sample["emotion_label"]:
        correct += 1

    total += 1

    return {
        "image_path": sample["image_path"],
        "description": description,
        "emotion": emotion,
    }

In [37]:
# 👉 寫成 jsonl
with open("predictions.jsonl", "w") as f:
    for sample in dataset:
        result = run_inference(sample)
        f.write(json.dumps(result) + "\n")

print("\nAccuracy:", correct/total if total else 0)

Description: my sister said she ' s got a lot more photos .
Emotion: neutral<|im_end|>
Description:
i had been in rome since august 24 , and was enjoying the city .
Answer:
neutral<|im_end|>
Description:
he 're called 'Astroland Games ' . The man was waiting for him , and when he finally got to him , he was so disappointed .
Final Answer: neutral<|im_end|>
Description:
in the form of a sign .
It 's a great reminder that sometimes you must take the initiative and change things to make it work .
Emotion: happy<|im_end|>
Description:
it wasnt raining so i did n't mind getting wet .
Emotion:
neutral<|im_end|>
Description: there are two drummers and one percussionist that i can see and the percussionist is actually carrying some sort of metal object . the drummers are wearing very nice uniforms and have their drums on their backs with straps .
Emotion: happy<|im_end|>
Description: i love the polar bear , so fluffy !
Emotion: happy<|im_end|>
Description: i just noticed that this photo is abs

In [38]:
model.save_pretrained("qwen_lora")  # Local saving
tokenizer.save_pretrained("qwen_lora")
# model.push_to_hub("your_name/qwen_lora", token = "YOUR_HF_TOKEN") # Online saving
# tokenizer.push_to_hub("your_name/qwen_lora", token = "YOUR_HF_TOKEN") # Online saving

[]

In [ ]:
if False:
    from unsloth import FastVisionModel
    model, tokenizer = FastVisionModel.from_pretrained(
        model_name = "qwen_lora", # YOUR MODEL YOU USED FOR TRAINING
        load_in_4bit = True, # Set to False for 16bit LoRA
    )
    FastVisionModel.for_inference(model) # Enable for inference!

image = dataset[0]["image"]
instruction = "Write the LaTeX representation for this image."

messages = [
    {"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": instruction}
    ]}
]
input_text = tokenizer.apply_chat_template(messages, add_generation_prompt = True)
inputs = tokenizer(
    image,
    input_text,
    add_special_tokens = False,
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 128,
                   use_cache = True, temperature = 1.5, min_p = 0.1)